In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_absolute_error
import numpy as np
%pip install kagglehub catboost lightgbm tqdm -q

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

import matplotlib.pyplot as plt
import numpy as np


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head(10)

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.hist(df["Delivery_Time"])
plt.show()


In [ ]:
# Task 1: Write your code here:

df = df.drop("Order_ID", axis=1)

# df = df.dropna()

# df = df.drop_duplicates()

# scaler = StandardScaler()
# features = df.drop("Delivery_Time", axis=1)
# df[features.columns] = scaler.fit_transform(features)

# df["Delivery_Time"].value_counts(normalize=True)



In [ ]:
# Task 2: Write your code here:
df.isnull().sum()
df = df.dropna()

In [ ]:
# Task 3: Write your code here:
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:

categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

le = LabelEncoder()
df['Weather'] = le.fit_transform(df['Weather'])
df['Traffic_Level'] = le.fit_transform(df['Traffic_Level'])
df['Time_of_Day'] = le.fit_transform(df['Time_of_Day'])
df['Vehicle_Type'] = le.fit_transform(df['Vehicle_Type'])



In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df.drop("Delivery_Time", axis=1))
df[df.drop("Delivery_Time", axis=1).columns] = scaled_features

In [ ]:
# Task 6: Write your code here:
df["Delivery_Time"].value_counts(normalize=True)

In [ ]:
# Task 1: Write your code here:

X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]


In [ ]:
# Task 2,3,4,5: Write your code here:

kf = KFold(n_splits=5, shuffle=True, random_state=42)

model = RandomForestRegressor(random_state=42)

mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

scores = cross_val_score(model, X, y, cv=kf, scoring=mae_scorer)

print(np.mean(-scores))

In [ ]:
# Task 1: Write your code here:
model.fit(X, y)

importances = model.feature_importances_
indices = np.argsort(importances)

plt.figure()
plt.title("Feature Importance")
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), X.columns[indices])
plt.show()


In [ ]:
# Task 2: Write your code here:

preds = model.predict(X)

plt.figure()
plt.hist(preds)
plt.title("Predicted Delivery Time Distribution")
plt.show()

In [ ]:
# Task Bonus: Write your code here:



X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf = RandomForestRegressor(random_state=42)
    cb = CatBoostRegressor(random_state=42, verbose=0)

    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)

    pred_rf = rf.predict(X_val)
    pred_cb = cb.predict(X_val)

    pred_avg = (pred_rf + pred_cb) / 2
    mae_scores.append(mean_absolute_error(y_val, pred_avg))

print(np.mean(mae_scores))
